# DisasterM3 — QLoRA Fine-Tuning of Qwen2.5-VL-7B
**Purpose:** Reproduce the fine-tuning pipeline from Wang et al. (NeurIPS 2025).  
**Hardware:** Kaggle T4 GPU (16 GB VRAM) — QLoRA 4-bit to fit in memory.  
**Reference:** Appendix B.3 of arXiv:2505.21089v2.

### How to run ("Run All" workflow)
1. **Prerequisites (once):** Add-ons → Secrets → `KAGGLE_USERNAME` + `KAGGLE_KEY`
   (toggles ON for this notebook); optional `HF_TOKEN` (write-scoped, from
   hf.co/settings/tokens) for HF Hub checkpoint/adapter backup; DisasterM3
   mirror dataset attached as input; GPU T4; internet ON.
2. **Fresh session:** click **Run All**. Cell 1 installs pinned deps and HALTS with
   a restart prompt → **Restart kernel** → **Run All** again. Cell 1 now skips
   installation and the whole pipeline runs through training.
3. Training self-stops after ~10.5 h (checkpoint saved); Cell 10b then pushes it
   to the private checkpoint dataset automatically as part of the same Run All.
4. **Resume session:** attach the `disasterm3-qlora-checkpoints` dataset as input,
   set `RESUME_CHECKPOINT_DATASET` in Cell 2, then repeat step 2.

### Deviations from Paper (logged in DEVIATIONS.md)
| Parameter | Paper (4×H100) | Ours (1×T4) |
|---|---|---|
| Precision | bf16 | fp16 (T4 lacks native bf16) |
| Quantization | None (full LoRA) | 4-bit NF4 (QLoRA) |
| Attention | FlashAttention-2 | SDPA (T4 is Turing cc7.5) |
| Per-device batch | 64 | 1–2 |
| Gradient accum. | 1 | 128–256 (to approximate global batch 256) |
| GPUs | 4 | 1 |
| Image resolution | dynamic (unbounded) | capped at ~512 vision tokens/image (`max_pixels`) |
| Loss target | (unspecified) | assistant response only (prompt + image tokens masked) |
| Seg. entries w/ path-like target | included(?) | dropped — text pipeline can't train on mask paths |
| Checkpointing | n/a | every 2 opt. steps + push to Kaggle Dataset (12 h cap) |

### What stays the same
LoRA rank=64, alpha=16, dropout=0.05, AdamW lr=2e-4, β₁=0.9, β₂=0.95,
cosine schedule, 1 epoch, vision encoder frozen.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# Pin exact versions to avoid Kaggle/torch conflicts.
# This cell is idempotent: if the pinned versions are already installed it
# does nothing and "Run All" proceeds straight into training. If it had to
# install anything, it HALTS the run and asks for a kernel restart — after
# restarting, click "Run All" again and it will skip installation.
import importlib.metadata as _md

PINS = {
    "torch": "2.4.1",
    "torchvision": "0.19.1",
    "torchaudio": "2.4.1",       # MUST match torch — unpinned torchaudio causes an ABI crash
    "transformers": "4.49.0",    # 4.46.x is too old for Qwen2_5_VLForConditionalGeneration
    "peft": "0.14.0",
    "bitsandbytes": "0.45.0",
    "accelerate": "1.2.1",
    "trl": "0.13.0",
    "qwen-vl-utils": "0.0.8",
}

_mismatched = []
for _pkg, _want in PINS.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _have = "not installed"
    if _have != _want:
        _mismatched.append(f"{_pkg}: {_have} → {_want}")

if _mismatched:
    print("⏳ Installing pinned versions:")
    for _m in _mismatched:
        print(f"   {_m}")
    !pip install -q \
        torch==2.4.1 \
        torchvision==0.19.1 \
        torchaudio==2.4.1 \
        transformers==4.49.0 \
        peft==0.14.0 \
        bitsandbytes==0.45.0 \
        accelerate==1.2.1 \
        trl==0.13.0 \
        qwen-vl-utils==0.0.8 \
        pillow \
        huggingface_hub[hf_xet]
    # Halt "Run All" here — imports must not happen before a kernel restart.
    raise SystemExit(
        "✓ Dependencies installed. RESTART THE KERNEL NOW "
        "(Run → Restart & clear cell outputs), then click 'Run All' again — "
        "this cell will detect the correct versions and skip installation."
    )

print("✓ All pinned versions already installed — proceeding.")

✓ All pinned versions already installed — proceeding.


In [2]:
import transformers
print(transformers.__version__)

4.49.0


In [3]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import os
import json
import torch
from pathlib import Path
from datetime import datetime

# ── Paths ──
# Using the exact path you found!
DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
print(f"✓ Using Kaggle Dataset mount: {DATA_ROOT}")

MANIFEST_PATH = DATA_ROOT / "train_release.json"
IMAGE_DIR = Path("/tmp/train_images")  # Will unzip here
BOX_IMAGE_DIR = Path("/tmp/box_train_images")
MASK_DIR = Path("/tmp/masks")

# ── Model ──
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── LoRA config (Appendix B.3) ──
LORA_RANK = 64
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# ── Training config ──
LEARNING_RATE = 2e-4
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.95
NUM_EPOCHS = 1
PER_DEVICE_BATCH = 1              # T4 constraint
GRADIENT_ACCUMULATION = 256       # Global batch ≈ 1 × 256 = 256
MAX_SEQ_LENGTH = 2048             # Qwen2.5-VL default
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

# ── Image token budget (T4 constraint — log in DEVIATIONS.md) ──
# Caps Qwen2.5-VL's dynamic resolution so a pre/post image pair (~2×512 tokens)
# always fits in MAX_SEQ_LENGTH without truncating image placeholder tokens.
# Also the single biggest throughput lever on a T4 (2-4× faster).
MIN_PIXELS = 256 * 28 * 28        # ≥ ~256 vision tokens per image
MAX_PIXELS = 512 * 28 * 28        # ≤ ~512 vision tokens per image

# ── Checkpointing (12-hour Kaggle session cap) ──
# SAVE_STEPS=2 optimizer steps × 256 grad-accum = 512 samples ≈ 25-50 min on T4.
# (The old SAVE_STEPS=50 meant ~11-21 h between saves — the first checkpoint
# would never be written before the session died.)
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
SAVE_STEPS = 2
LOGGING_STEPS = 1                 # 1 optimizer step = 256 samples; log every step
TIME_LIMIT_HOURS = 11           # Save + stop cleanly before Kaggle's 12 h wall

# ── Cross-session resume ──
# After the first session: push checkpoints to a private Kaggle Dataset
# (Cell 10b), attach it as input next session, and point this at its mount path,
# e.g. "/kaggle/input/disasterm3-qlora-checkpoints". None = start fresh.
RESUME_CHECKPOINT_DATASET = None

# ── Output ──
OUTPUT_DIR = "/kaggle/working/qwen25vl_disasterm3_qlora"
ADAPTER_NAME = f"disasterm3_qwen25vl7b_qlora_r{LORA_RANK}_a{LORA_ALPHA}"

print(f"✓ Config loaded")
print(f"  Model: {MODEL_ID}")
print(f"  LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  Effective batch size: {PER_DEVICE_BATCH} × {GRADIENT_ACCUMULATION} = {PER_DEVICE_BATCH * GRADIENT_ACCUMULATION}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")


✓ Using Kaggle Dataset mount: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct
✓ Config loaded
  Model: Qwen/Qwen2.5-VL-7B-Instruct
  LoRA: rank=64, alpha=16, dropout=0.05
  Effective batch size: 1 × 256 = 256
  Device: Tesla T4
  VRAM: 15.6 GB


In [4]:
# # ── Cell 3: Unzip images ──────────────────────────────────────────────────
# import zipfile
# import time

# def unzip_if_needed(zip_path, dest_dir, label):
#     """Unzip only if destination doesn't already exist (resume-safe)."""
#     if dest_dir.exists() and any(dest_dir.iterdir()):
#         count = sum(1 for _ in dest_dir.rglob("*") if _.is_file())
#         print(f"✓ {label}: already unpacked ({count:,} files)")
#         return
    
#     dest_dir.mkdir(parents=True, exist_ok=True)
#     print(f"⏳ Unpacking {label}...")
#     t0 = time.time()
#     with zipfile.ZipFile(zip_path, "r") as zf:
#         zf.extractall(dest_dir.parent)
#     elapsed = time.time() - t0
#     count = sum(1 for _ in dest_dir.rglob("*") if _.is_file())
#     print(f"✓ {label}: unpacked {count:,} files in {elapsed:.0f}s")

# # Main training images (29.7 GB — this takes ~5-10 min)
# train_zip = DATA_ROOT / "train_images.zip"
# if train_zip.exists():
#     unzip_if_needed(train_zip, IMAGE_DIR, "train_images")
# else:
#     print(f"⚠ {train_zip} not found — skipping main images")

# # Box images for ORR/BBR tasks (2.96 GB)
# box_zip = DATA_ROOT / "box_train_images.zip"
# if box_zip.exists():
#     unzip_if_needed(box_zip, BOX_IMAGE_DIR, "box_train_images")
# else:
#     print(f"⚠ {box_zip} not found — skipping box images")

# # Masks for segmentation (173 MB)
# mask_zip = DATA_ROOT / "masks.zip"
# if mask_zip.exists():
#     unzip_if_needed(mask_zip, MASK_DIR, "masks")
# else:
#     print(f"⚠ {mask_zip} not found — skipping masks")

## 2. Load & Prepare Training Data

In [5]:
# ── Cell 4: Load manifest and build chat dataset ─────────────────────────
from PIL import Image

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Audit: segmentation training_answer (plan addendum item 6) ────────────
# For Referring Expression Segmentation, ground_truth is a MASK FILE PATH.
# If training_answer were empty and we fell back to ground_truth, we would be
# training the model to emit path strings for 40% of the data. Audit first.
PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

def _answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

seg_entries = [e for e in raw_data if e.get("task") == "Referring Expression Segmentation"]
seg_empty = sum(1 for e in seg_entries if not _answer_text(e))
seg_pathlike = sum(1 for e in seg_entries if _answer_text(e).lower().endswith(PATH_SUFFIXES))
print(f"\nSegmentation audit: {len(seg_entries):,} entries | "
      f"empty training_answer: {seg_empty:,} | path-like: {seg_pathlike:,}")
if seg_entries:
    ex = seg_entries[0]
    print(f"  sample training_answer: {_answer_text(ex)[:150]!r}")
    print(f"  sample ground_truth:    {str(ex.get('ground_truth', ''))[:150]!r}")

def resolve_image_path(rel_path):
    """Resolve a manifest-relative path to an absolute path on disk."""
    if not rel_path:
        return None
    # Normalize backslashes
    rel_path = rel_path.replace("\\", "/")
    
    # Extract just the raw filename (e.g. "image_0.png")
    filename = Path(rel_path).name
    
    # Try all possible locations, explicitly checking Kaggle's double-nested folders
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "box_train_images" / rel_path,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "box_train_images" / "box_train_images" / filename,
        DATA_ROOT / "masks" / "masks" / filename,
        Path("/tmp") / rel_path
    ]
    
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
            
    return None


def entry_to_messages(entry):
    """Convert a manifest entry to Qwen2.5-VL chat messages format."""
    task = entry.get("task", "")
    
    # Build user content with images
    user_content = []
    
    if "image_path" in entry:
        # Relational reasoning: single post-disaster image with bounding boxes
        img_path = resolve_image_path(entry["image_path"])
        if img_path:
            user_content.append({"type": "image", "image": f"file://{img_path}"})
    else:
        # All other tasks: pre + post disaster pair
        pre_path = resolve_image_path(entry.get("pre_image_path", ""))
        post_path = resolve_image_path(entry.get("post_image_path", ""))
        if pre_path:
            user_content.append({"type": "image", "image": f"file://{pre_path}"})
        if post_path:
            user_content.append({"type": "image", "image": f"file://{post_path}"})
    
    # Skip entries where images couldn't be resolved
    if not any(c.get("type") == "image" for c in user_content):
        return None
    
    # Add prompt text
    prompt = entry.get("prompts", "")
    if isinstance(prompt, list):
        prompt = prompt[0]
    user_content.append({"type": "text", "text": prompt})
    
    # Get the training answer.
    # NO ground_truth fallback: for segmentation entries ground_truth is a mask
    # file path — training on it would teach the model to emit path strings.
    # Entries with an empty or path-like target are dropped (logged below and
    # recorded in DEVIATIONS.md).
    answer = _answer_text(entry)
    if not answer or answer.lower().endswith(PATH_SUFFIXES):
        return None
    
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    ]
    return messages

# Convert all entries
print("Converting manifest entries to chat format...")
chat_dataset = []
from collections import Counter
task_counts = Counter()
skip_counts = Counter()

for entry in raw_data:
    msgs = entry_to_messages(entry)
    task = entry.get("task", "unknown")
    if msgs is not None:
        chat_dataset.append({"messages": msgs, "task": task})
        task_counts[task] += 1
    else:
        skip_counts[task] += 1

print(f"\n✓ Converted: {len(chat_dataset):,} entries")
print(f"  Skipped (missing images or unusable answer): {sum(skip_counts.values()):,}")
if skip_counts:
    print(f"\n  Skips per task (record in DEVIATIONS.md):")
    for task, count in skip_counts.most_common():
        print(f"    {task:<45} {count:>6,}")
print(f"\nPer-task breakdown (kept):")
for task, count in task_counts.most_common():
    print(f"  {task:<45} {count:>6,}")


Loaded 92,968 entries from manifest

Segmentation audit: 37,204 entries | empty training_answer: 37,204 | path-like: 0
  sample training_answer: ''
  sample ground_truth:    'masks\\flooding_mask\\hurricane_florence_00000029.png'
Converting manifest entries to chat format...

✓ Converted: 55,764 entries
  Skipped (missing images or unusable answer): 37,204

  Skips per task (record in DEVIATIONS.md):
    Referring Expression Segmentation             37,204

Per-task breakdown (kept):
  Building Damage Counting                      14,531
  Disaster Bearing Bodies Recognition            7,766
  disaster caption                               7,766
  disaster restoration advice                    7,765
  Road Damage Counting                           7,337
  Disaster Scene Recognition                     7,090
  relational reasoning                           1,882
  Disaster Type Recognition                      1,627


## 3. Load Model with QLoRA

In [7]:
# ── Cell 5: Load Qwen2.5-VL-7B with 4-bit quantization ───────────────────
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # fp16 (T4 has no native bf16)
    bnb_4bit_use_double_quant=True,
)

print(f"⏳ Loading {MODEL_ID} in 4-bit...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # T4 can't use flash_attention_2
    torch_dtype=torch.float16,
)

# Pixel budget bounds vision tokens (~256-512 per image) so a pre/post pair
# plus prompt+answer fits in MAX_SEQ_LENGTH without cutting image tokens.
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print(f"✓ Model loaded")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

⏳ Loading Qwen/Qwen2.5-VL-7B-Instruct in 4-bit...


model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

✓ Model loaded
  Parameters: 4,692,257,792
  Trainable before LoRA: 0


In [8]:
# ── Cell 6: Apply LoRA adapters ───────────────────────────────────────────
# Target: LLM component ONLY (vision encoder frozen) — matches paper's recipe.
# IMPORTANT: a plain name list like ["gate_proj", ...] ALSO matches the vision
# tower's MLP layers (Qwen2.5-VL's ViT blocks use the same gate/up/down_proj
# names), silently adding ~29M trainable params inside the "frozen" vision
# encoder. The regex below anchors matching to the language model's layers.

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    # str target_modules is treated as a regex (full match on module path):
    # matches model.layers.N.self_attn.{q,k,v,o}_proj and
    #         model.layers.N.mlp.{gate,up,down}_proj — never visual.blocks.*
    target_modules=(
        r"^model\.layers\.\d+\."
        r"(self_attn\.(q_proj|k_proj|v_proj|o_proj)"
        r"|mlp\.(gate_proj|up_proj|down_proj))$"
    ),
    task_type="CAUSAL_LM",
    bias="none",
)

model = get_peft_model(model, lora_config)

# Guard: fail loudly if any LoRA module ended up in the vision tower.
_lora_on_vision = [
    n for n, _ in model.named_modules()
    if "visual" in n and "lora_" in n
]
assert not _lora_on_vision, (
    f"LoRA attached to vision tower (violates frozen-vision recipe): "
    f"{_lora_on_vision[:5]}"
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA adapters applied (LLM only — vision tower verified LoRA-free)")
print(f"  Trainable parameters: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"  Expected (28 LLM layers, r=64): 161,480,704")
print(f"  Total parameters:     {total:,}")
model.print_trainable_parameters()

✓ LoRA adapters applied (LLM only — vision tower verified LoRA-free)
  Trainable parameters: 161,480,704 (3.33%)
  Expected (28 LLM layers, r=64): 161,480,704
  Total parameters:     4,853,738,496
trainable params: 161,480,704 || all params: 8,453,647,360 || trainable%: 1.9102


## 4. Training Loop

In [9]:
# ── Cell 7: Set up SFTTrainer ─────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
import torch.utils.data

# Keep examples as plain Python dicts. Do NOT use datasets.Dataset.from_list():
# Arrow unifies the schema of the nested content items and injects `image: None`
# into text items, which crashes process_vision_info on the first batch
# (the AttributeError observed in the first training attempt).
class ChatListDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = ChatListDataset(chat_dataset)

# SFT training config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    adam_beta1=ADAM_BETA1,
    adam_beta2=ADAM_BETA2,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,                         # T4: fp16 only
    bf16=False,                        # T4: no native bf16
    gradient_checkpointing=True,       # Save VRAM
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,                # Keep last 3 checkpoints
    report_to="none",                  # No wandb on Kaggle
    dataloader_num_workers=2,
    remove_unused_columns=False,
    max_grad_norm=1.0,                 # Gradient clipping (important for fp16)
    dataset_text_field=None,           # We use messages format
    dataset_kwargs={"skip_prepare_dataset": True},
)

print("✓ Training config ready")
print(f"  Steps per epoch: ~{len(chat_dataset) // (PER_DEVICE_BATCH * GRADIENT_ACCUMULATION)}")
print(f"  Total optimizer steps: ~{len(chat_dataset) // (PER_DEVICE_BATCH * GRADIENT_ACCUMULATION) * NUM_EPOCHS}")

✓ Training config ready
  Steps per epoch: ~217
  Total optimizer steps: ~217


In [10]:
# ── Cell 8: Custom data collator for Qwen2.5-VL ──────────────────────────
from qwen_vl_utils import process_vision_info

# Token ids used for label masking
IMAGE_PAD_ID = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
VISION_START_ID = processor.tokenizer.convert_tokens_to_ids("<|vision_start|>")
VISION_END_ID = processor.tokenizer.convert_tokens_to_ids("<|vision_end|>")
# "<|im_start|>assistant\n" marks where the response begins in Qwen's template
ASSISTANT_MARKER_IDS = processor.tokenizer(
    "<|im_start|>assistant\n", add_special_tokens=False
)["input_ids"]


def _strip_none(content):
    """Drop None-valued keys from content items (defensive vs Arrow-style
    schema padding, which crashes process_vision_info)."""
    return [{k: v for k, v in item.items() if v is not None} for item in content]


def _mask_prompt_tokens(labels_row):
    """Mask everything up to and including the last assistant marker, so loss
    covers only the assistant response (standard SFT-on-responses)."""
    ids = labels_row.tolist()
    m = ASSISTANT_MARKER_IDS
    for start in range(len(ids) - len(m), -1, -1):
        if ids[start:start + len(m)] == m:
            labels_row[: start + len(m)] = -100
            return True
    return False


def collate_fn(examples):
    """Collator for Qwen2.5-VL multi-image inputs with assistant-only loss."""
    texts = []
    all_images = []  # flat list, in example order — the processor matches
                     # images to <image> placeholders in order across the batch

    for example in examples:
        messages = [
            {"role": msg["role"], "content": _strip_none(msg["content"])}
            for msg in example["messages"]
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)

        image_inputs, _ = process_vision_info(messages)
        if image_inputs:
            all_images.extend(image_inputs)

    # Truncation is safe now: image tokens sit at the START of the sequence and
    # the processor's max_pixels cap bounds them (~1024 for a pre/post pair),
    # so truncation can only ever trim answer text, never image placeholders.
    batch = processor(
        text=texts,
        images=all_images if all_images else None,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    # Never compute LM loss on vision positions
    labels[labels == IMAGE_PAD_ID] = -100
    labels[labels == VISION_START_ID] = -100
    labels[labels == VISION_END_ID] = -100
    # Assistant-only loss: mask the system/user turn in every row
    for row in labels:
        _mask_prompt_tokens(row)
    batch["labels"] = labels

    return batch

print("✓ Custom collator defined (assistant-only loss, image tokens masked)")

✓ Custom collator defined (assistant-only loss, image tokens masked)


In [11]:
# ── Cell 8.5: Restore latest checkpoint from HF (run before Cell 9) ───────────────
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
import shutil, os, glob

HF_REPO = "AbrarAlam/disasterm3-qwen25vl7b-qlora"
CKPT_NAME = "checkpoint-102"  # update each session to latest on HF

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
local = snapshot_download(
    repo_id=HF_REPO,
    repo_type="model",
    token=hf_token,
    allow_patterns=f"checkpoints/{CKPT_NAME}/**",
)

src = os.path.join(local, "checkpoints", CKPT_NAME)
dst = os.path.join(OUTPUT_DIR, CKPT_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f"✓ Restored {CKPT_NAME} → {dst}")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

checkpoints/checkpoint-102/rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

dataset-metadata.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

...

trainer_state.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

checkpoints/checkpoint-102/training_args(…):   0%|          | 0.00/5.62k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✓ Restored checkpoint-102 → /kaggle/working/qwen25vl_disasterm3_qlora/checkpoint-102


In [12]:
# ── Cell 9: Resume from checkpoint if available ─────────────────────────
import shutil

# Step 1: fresh Kaggle sessions start with an empty /kaggle/working, so a
# checkpoint saved last session must be restored from the attached Kaggle
# Dataset (set RESUME_CHECKPOINT_DATASET in Cell 2 after attaching it).
if RESUME_CHECKPOINT_DATASET and os.path.exists(RESUME_CHECKPOINT_DATASET):
    src_ckpts = [
        d for d in os.listdir(RESUME_CHECKPOINT_DATASET)
        if d.startswith("checkpoint-")
        and os.path.isdir(os.path.join(RESUME_CHECKPOINT_DATASET, d))
    ]
    if src_ckpts:
        latest_src = max(src_ckpts, key=lambda x: int(x.split("-")[1]))
        src = os.path.join(RESUME_CHECKPOINT_DATASET, latest_src)
        dst = os.path.join(OUTPUT_DIR, latest_src)
        if not os.path.exists(dst):
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            print(f"⏳ Restoring {latest_src} from attached checkpoint dataset...")
            shutil.copytree(src, dst)
            print(f"✓ Restored to {dst}")
        else:
            print(f"✓ {latest_src} already present in {OUTPUT_DIR}")
    else:
        print(f"⚠ No checkpoint-* folders found in {RESUME_CHECKPOINT_DATASET}")

# Step 2: scan OUTPUT_DIR for the latest checkpoint
resume_from = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = [
        d for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(OUTPUT_DIR, d))
    ]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
        resume_from = os.path.join(OUTPUT_DIR, latest)
        print(f"✓ Resuming from checkpoint: {resume_from}")
    else:
        print("No existing checkpoints found — starting fresh")
else:
    print("No output directory found — starting fresh")

✓ Resuming from checkpoint: /kaggle/working/qwen25vl_disasterm3_qlora/checkpoint-102


In [14]:
# ── Cell 10: Train! ───────────────────────────────────────────────────────
import time
from transformers import TrainerCallback

class TimeLimitCallback(TrainerCallback):
    """Write a final checkpoint and stop cleanly before Kaggle's 12 h wall,
    leaving time to persist it (Cell 10b) instead of being killed mid-step."""
    def __init__(self, max_hours):
        self.deadline = time.time() + max_hours * 3600
    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            print(f"⏰ Time budget reached at optimizer step {state.global_step} "
                  f"— saving checkpoint and stopping.")
            control.should_save = True
            control.should_training_stop = True
        return control

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    processing_class=processor.tokenizer,
    callbacks=[TimeLimitCallback(TIME_LIMIT_HOURS)],
)

print(f"⏳ Starting training at {datetime.now().strftime('%H:%M:%S')}...")
print(f"   This will take several hours on a T4.")
print(f"   Checkpoints saved every {SAVE_STEPS} steps to {OUTPUT_DIR}")
print()

trainer.train(resume_from_checkpoint=resume_from)

print(f"\n✓ Training complete at {datetime.now().strftime('%H:%M:%S')}")

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


⏳ Starting training at 08:25:38...
   This will take several hours on a T4.
   Checkpoints saved every 2 steps to /kaggle/working/qwen25vl_disasterm3_qlora



ValueError: Can't find a valid checkpoint at /kaggle/working/qwen25vl_disasterm3_qlora/checkpoint-102

In [15]:
# ── Cell 10b: Persist latest checkpoint across sessions ──────────────────
# /kaggle/working is exported ONLY when a "Save Version" run completes. If the
# interactive session is killed at the 12 h wall, everything here is lost.
# This cell pushes the latest checkpoint to a private Kaggle Dataset.
# Requires: Add-ons → Secrets → KAGGLE_USERNAME and KAGGLE_KEY (same secrets
# already used by the mirror notebook), with their toggles enabled for THIS
# notebook.
# Next session: attach that dataset as input and set RESUME_CHECKPOINT_DATASET.
import glob
import subprocess

ckpts = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
    key=lambda p: int(p.split("-")[-1]),
)
if not ckpts:
    print("No checkpoints found — nothing to persist.")
else:
    latest = ckpts[-1]
    print(f"Latest checkpoint: {latest}")
    try:
        # Same auth pattern as the mirror notebook (proven working there):
        # KAGGLE_USERNAME + KAGGLE_KEY secrets → env vars read by the kaggle CLI.
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")

        username = os.environ["KAGGLE_USERNAME"]
        ds_slug = f"{username}/disasterm3-qlora-checkpoints"
        with open(os.path.join(latest, "dataset-metadata.json"), "w") as f:
            json.dump({
                "title": "disasterm3-qlora-checkpoints",
                "id": ds_slug,
                "licenses": [{"name": "CC0-1.0"}],
            }, f)

        # First run creates the dataset; later runs add a new version.
        r = subprocess.run(
            ["kaggle", "datasets", "create", "-p", latest, "--dir-mode", "zip"],
            capture_output=True, text=True,
        )
        if "already exists" in (r.stdout + r.stderr).lower():
            r = subprocess.run(
                ["kaggle", "datasets", "version", "-p", latest,
                 "-m", f"{os.path.basename(latest)}", "--dir-mode", "zip"],
                capture_output=True, text=True,
            )
        print((r.stdout or r.stderr)[-2000:])
        print(f"✓ Pushed {os.path.basename(latest)} to kaggle.com/datasets/{ds_slug}")
        print(f"  Next session: attach it as input and set")
        print(f"  RESUME_CHECKPOINT_DATASET = '/kaggle/input/disasterm3-qlora-checkpoints'")
    except Exception as e:
        print(f"⚠ Kaggle API push failed ({e}).")
        print("  Fallback options:")
        print("  1. Run 'Save Version' so /kaggle/working is exported with the notebook.")
        print("  2. Download the checkpoint folder manually before the session ends.")

    # Disaster backup: also mirror the checkpoint to a private HF Hub repo.
    # Optional — requires an "HF_TOKEN" secret (write-scoped) toggled on for
    # this notebook. Skipped silently if not configured.
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        hf_user = api.whoami()["name"]
        hf_repo = f"{hf_user}/disasterm3-qwen25vl7b-qlora"
        api.create_repo(hf_repo, private=True, exist_ok=True)
        api.upload_folder(
            folder_path=latest,
            repo_id=hf_repo,
            path_in_repo=f"checkpoints/{os.path.basename(latest)}",
            commit_message=f"backup {os.path.basename(latest)}",
        )
        print(f"✓ Mirrored to HF Hub: {hf_repo}/checkpoints/{os.path.basename(latest)}")
    except Exception as e:
        print(f"ℹ HF Hub mirror skipped ({e}).")

Latest checkpoint: /kaggle/working/qwen25vl_disasterm3_qlora/checkpoint-118
Starting upload for file optimizer.pt
Upload successful: optimizer.pt (1GB)
Starting upload for file adapter_model.safetensors
Upload successful: adapter_model.safetensors (616MB)
Starting upload for file tokenizer_config.json
Upload successful: tokenizer_config.json (6KB)
Starting upload for file added_tokens.json
Upload successful: added_tokens.json (605B)
Starting upload for file training_args.bin
Upload successful: training_args.bin (5KB)
Starting upload for file trainer_state.json
Upload successful: trainer_state.json (21KB)
Starting upload for file README.md
Upload successful: README.md (5KB)
Starting upload for file scheduler.pt
Upload successful: scheduler.pt (1KB)
Starting upload for file rng_state.pth
Upload successful: rng_state.pth (14KB)
Starting upload for file merges.txt
Upload successful: merges.txt (2MB)
Starting upload for file vocab.json
Upload successful: vocab.json (3MB)
Starting upload for

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Mirrored to HF Hub: AbrarAlam/disasterm3-qwen25vl7b-qlora/checkpoints/checkpoint-118


In [16]:
# ── Cell 10c: Loss Curve ──────────────────────────────────────────────────
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

log_history = []

# 1. Try to get logs from live RAM (if we just finished training)
if 'trainer' in globals() and hasattr(trainer, 'state') and getattr(trainer.state, 'log_history', []):
    log_history = trainer.state.log_history
# 2. Fallback: Load from the latest checkpoint on disk
elif os.path.exists(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-') and os.path.isdir(os.path.join(OUTPUT_DIR, d))]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.split('-')[1]))
        state_file = os.path.join(OUTPUT_DIR, latest, 'trainer_state.json')
        if os.path.exists(state_file):
            print(f"Loading logs from {state_file}...")
            with open(state_file, 'r', encoding='utf-8') as f:
                state_data = json.load(f)
                log_history = state_data.get('log_history', [])

if log_history:
    df = pd.DataFrame(log_history)
    if 'loss' in df.columns:
        df = df.dropna(subset=['loss'])
        plt.figure(figsize=(10, 5))
        plt.plot(df['step'], df['loss'], label='Training Loss', color='blue', linewidth=2)
        plt.xlabel('Steps')
        plt.ylabel('Loss')
        plt.title('QLoRA Fine-Tuning Loss')
        plt.grid(True)
        plt.legend()
        loss_plot_path = os.path.join(OUTPUT_DIR, 'loss_curve.png')
        plt.savefig(loss_plot_path)
        print(f"✓ Saved loss curve to {loss_plot_path}")
        plt.show()
    else:
        print("No 'loss' column found in logs.")
else:
    print("No training logs found — run training first (or restore a checkpoint).")


No training logs found — run training first (or restore a checkpoint).


## 5. Save Adapter Checkpoint

In [17]:
# ── Cell 11: Save the trained LoRA adapter ───────────────────────────────
adapter_path = os.path.join(OUTPUT_DIR, ADAPTER_NAME)
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)

print(f"✓ Adapter saved to: {adapter_path}")
print(f"  Contents:")
for f in sorted(os.listdir(adapter_path)):
    size = os.path.getsize(os.path.join(adapter_path, f)) / 1e6
    print(f"    {f:<40} {size:.1f} MB")

# Log training summary
summary = {
    "model": MODEL_ID,
    "adapter_name": ADAPTER_NAME,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "per_device_batch": PER_DEVICE_BATCH,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "effective_batch_size": PER_DEVICE_BATCH * GRADIENT_ACCUMULATION,
    "precision": "fp16 + NF4 QLoRA",
    "attention": "sdpa",
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "loss_target": "assistant response only (prompt/image tokens masked)",
    "training_entries": len(chat_dataset),
    "skipped_entries": dict(skip_counts),
    "timestamp": datetime.now().isoformat(),
}
with open(os.path.join(adapter_path, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Training summary written to {adapter_path}/training_summary.json")

# Bundle the aggregated loss history into the adapter folder so the HF repo's
# adapter is self-documenting: loss_curve.png (from Cell 10c) plus the latest
# checkpoint's trainer_state.json (full log_history from step 0, all sessions).
import glob
import shutil

curve_src = os.path.join(OUTPUT_DIR, "loss_curve.png")
if os.path.exists(curve_src):
    shutil.copy2(curve_src, os.path.join(adapter_path, "loss_curve.png"))
    print("✓ Bundled loss_curve.png into adapter folder")
else:
    print("ℹ No loss_curve.png found (run Cell 10c to generate it)")

_ckpts = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
    key=lambda p: int(p.split("-")[-1]),
)
if _ckpts:
    state_src = os.path.join(_ckpts[-1], "trainer_state.json")
    if os.path.exists(state_src):
        shutil.copy2(state_src, os.path.join(adapter_path, "trainer_state.json"))
        print(f"✓ Bundled trainer_state.json (from {os.path.basename(_ckpts[-1])}, "
              f"full loss history) into adapter folder")

# Push the adapter to a private HF Hub repo (durable home for the deliverable;
# also where the eval phase will pull it from for merge_and_unload + vLLM).
# Optional — requires the "HF_TOKEN" secret toggled on for this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    hf_user = api.whoami()["name"]
    hf_repo = f"{hf_user}/disasterm3-qwen25vl7b-qlora"
    api.create_repo(hf_repo, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=adapter_path,
        repo_id=hf_repo,
        path_in_repo=ADAPTER_NAME,
        commit_message=f"adapter snapshot @ {datetime.now().isoformat()}",
    )
    print(f"✓ Adapter pushed to HF Hub: {hf_repo}/{ADAPTER_NAME}")
except Exception as e:
    print(f"ℹ HF Hub adapter push skipped ({e}).")
    print("  To enable: create a write-scoped token at hf.co/settings/tokens,")
    print("  add it as a Kaggle secret named HF_TOKEN, and toggle it on.")

✓ Adapter saved to: /kaggle/working/qwen25vl_disasterm3_qlora/disasterm3_qwen25vl7b_qlora_r64_a16
  Contents:
    README.md                                0.0 MB
    adapter_config.json                      0.0 MB
    adapter_model.safetensors                646.0 MB
    added_tokens.json                        0.0 MB
    chat_template.json                       0.0 MB
    merges.txt                               1.7 MB
    preprocessor_config.json                 0.0 MB
    special_tokens_map.json                  0.0 MB
    tokenizer.json                           11.4 MB
    tokenizer_config.json                    0.0 MB
    vocab.json                               2.8 MB

✓ Training summary written to /kaggle/working/qwen25vl_disasterm3_qlora/disasterm3_qwen25vl7b_qlora_r64_a16/training_summary.json
ℹ No loss_curve.png found (run Cell 10c to generate it)
✓ Bundled trainer_state.json (from checkpoint-118, full loss history) into adapter folder


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Adapter pushed to HF Hub: AbrarAlam/disasterm3-qwen25vl7b-qlora/disasterm3_qwen25vl7b_qlora_r64_a16


## 6. Quick Sanity Check
Run a single inference to verify the adapter loaded correctly.

In [18]:
# ── Cell 12: Sanity check inference ───────────────────────────────────────
# Pick a random training example and check the model can generate something
import random

test_entry = random.choice(chat_dataset)
test_messages = test_entry["messages"]

# Only keep the user message (remove assistant for generation)
user_msg = [test_messages[0]]

text = processor.apply_chat_template(
    user_msg,
    tokenize=False,
    add_generation_prompt=True,
)
image_inputs, _ = process_vision_info(user_msg)

inputs = processor(
    text=[text],
    images=image_inputs,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

# Decode only the new tokens
output_text = processor.batch_decode(
    generated[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)[0]

print(f"Task: {test_entry['task']}")
print(f"Expected: {test_messages[1]['content'][0]['text'][:200]}...")
print(f"\nGenerated: {output_text[:200]}...")
print(f"\n✓ Sanity check passed — model generates text after fine-tuning.")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:92: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Task: disaster restoration advice
Expected: IMMEDIATE_RECOVERY: Immediate recovery actions should focus on draining floodwaters from inundated agricultural fields and vegetation zones to prevent further crop damage and ecological degradation. T...

Generated: IMMEDIATE_RECOVERY: Immediate recovery actions should focus on draining floodwaters from the inundated agricultural fields to prevent further crop loss and supporting farmers with emergency seed suppl...

✓ Sanity check passed — model generates text after fine-tuning.
